In [ ]:
# Cell 1: Install pure-Python spell-checker and NLP tools
!pip install -q pandas spylls indic-nlp-library nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 5.4 MB/s eta 0:00:00


In [ ]:
# Cell 2: Download Standard Dictionaries and our ~1.77L word dataset
import urllib.request
import os
import pandas as pd
import nltk

print("1. Downloading NLTK English word list...")
nltk.download('words', quiet=True)

print("2. Downloading standard Hindi dictionary files (hi_IN)...")
if not os.path.exists("hi_IN.aff"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/LibreOffice/dictionaries/master/hi_IN/hi_IN.aff", "hi_IN.aff")
if not os.path.exists("hi_IN.dic"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/LibreOffice/dictionaries/master/hi_IN/hi_IN.dic", "hi_IN.dic")

print("3. Loading Question 3 Dataset from Google Sheets...")
sheet_url = "https://docs.google.com/spreadsheets/d/17DwCAx6Tym5Nt7eOni848np9meR-TIj7uULMtYcgQaw/export?format=csv"
df = pd.read_csv(sheet_url)

# Rename the column for easier access and remove random empty spaces
df.columns = ['Word']
df = df.dropna(subset=['Word'])
df['Word'] = df['Word'].astype(str).str.strip()

print(f"Dataset successfully loaded. Total unique words: {len(df)}")


1. Downloading NLTK English word list...
2. Downloading standard Hindi dictionary files (hi_IN)...
3. Loading Question 3 Dataset from Google Sheets...
Dataset successfully loaded. Total unique words: 177508


In [ ]:
# Cell 3: Define our Spell Checking Pipeline Logic
import re
from spylls.hunspell import Dictionary
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator
from nltk.corpus import words

# Initialize the dictionaries for fast lookup
hi_dict = Dictionary.from_files('hi_IN')
en_words = set(words.words())

def classify_spelling(word):
    # Fallback to avoid errors on completely weird strings
    if not isinstance(word, str) or len(word) == 0:
        return ("incorrect spelling", "High", "Empty or non-string word")

    # RULE 1: Standard Dictionary Lookup
    try:
        if hi_dict.lookup(word):
            return ("correct spelling", "High", "Found in standard Hindi vocabulary")
    except Exception:
        pass

    # RULE 2: Phonotactic / Formatting Constraints (Linguistic Reasoning)
    # Examples of impossible grammar: starts with a matra, or two matras back to back
    invalid_patterns = [
        r'^[ािीुूृेैोौंःँ]+', # starts with a vowel sign
        r'[ािीुूृेैोौंःँ]{2,}' # two vowel signs consecutively
    ]
    for pattern in invalid_patterns:
        if re.search(pattern, word):
            return ("incorrect spelling", "High", "Invalid Hindi character sequence (e.g. double matras)")

    # RULE 3: English Transliteration Check (e.g., "कंप्यूटर" -> "computer")
    # IndicNLP maps Devanagari back to Latin letters so we can check if it sounds like English
    try:
        # transliterate to ASCII
        transliterated = UnicodeIndicTransliterator.transliterate(word, "hi", "itrans").lower()
        # Very simple heuristic: if the exact root maps to an English NLTK word
        if transliterated in en_words:
            return ("correct spelling", "Medium", "Likely an English word transliterated to Hindi")
    except Exception:
        pass

    # FALLBACK: We don't recognize it at all
    return ("incorrect spelling", "Low", "Not found in Hindi dictionary and no obvious English equivalent")

print("Pipeline compiled successfully!")


Pipeline compiled successfully!


In [ ]:
# Cell 4: Run the Classification!
print("Starting classification over 1,77,000 words... please wait!")

# Apply our logic to every word
results = df['Word'].apply(classify_spelling)

# Split our tuples out into their new columns
df['Classification'] = [res[0] for res in results]
df['Confidence Score'] = [res[1] for res in results]
df['Reason'] = [res[2] for res in results]

print("Classification Complete!")

# Deliverable A: Final number of unique correctly spelled words
total_correct = len(df[df['Classification'] == 'correct spelling'])
print(f"\nFinal Number of Correctly Spelled Words: {total_correct}")


Starting classification over 1,77,000 words... please wait!


In [ ]:
# Cell 5: Create your Google Sheet deliverable and isolate samples for review
# The deliverables require: a sheet with [Word, Classification]
deliverable_df = df[['Word', 'Classification']]
deliverable_df.to_csv("Question3_Spelling_Classification.csv", index=False)
print("✅ Saved main deliverable as 'Question3_Spelling_Classification.csv'.")

# Let's also isolate the "Low Confidence" errors for your manual review (Question 3c)
low_confidence_df = df[df['Confidence Score'] == 'Low']

# Sample 50 random words from the Low bucket for Question 3(c)
if not low_confidence_df.empty:
    sample_size = min(50, len(low_confidence_df))
    low_confidence_sample = low_confidence_df.sample(sample_size, random_state=42)
    low_confidence_sample.to_csv("Low_Confidence_Sample_For_Review.csv", index=False)
    print(f"✅ Saved 'Low_Confidence_Sample_For_Review.csv' (Contains {sample_size} words for you to manually review for Part c & d)")
